In [106]:
# import libraries
import pandas as pd
from datetime import datetime
from time_diff_function import timediff

In [107]:
start_time  = datetime.now()
print(f'Amending \eagle\.. files before StatPro run ...')

Amending \eagle\.. files before StatPro run ...


In [108]:
# get instruments.csv and Holdings.csv data in eagle folder
eagle_path   = r'\\Pim-cpt-statpro\Plugins\Profiles\PIM\DataStage\Data\eagle'
holdings     = pd.read_csv(eagle_path + r'\Holdings.csv')
instruments  = pd.read_csv(eagle_path + r'\Instruments.csv', on_bad_lines = 'skip')
transactions = pd.read_csv(eagle_path + r'\Transactions.csv')
# in case there's a tokenization error:
 #https://saturncloud.io/blog/how-to-fix-python-pandas-error-tokenizing-data/#:~:text=Fixing%20the%20Error,
 #-Here%20are%20some&text=The%20first%20step%20is%20to,it%20can%20be%20read%20properly

In [110]:
# get lookups for missing Instruments.csv and Holdings.csv data
py_reports  = r'\\PIM-CPT-FS.prescient.local\PIM-Documents$\Investment Operations\GRC\Compliance\Daily\py_reports.xlsm'
currency    = pd.read_excel(py_reports, sheet_name = 'statpro', usecols = "A:B")
# for "NA" passing as NaN - https://stackoverflow.com/questions/41417214/prevent-pandas-from-reading-na-as-nan

# get issuer names and shorter issuer name replacements
issuer_name  = pd.read_excel(py_reports, sheet_name = 'statpro', usecols = "E:F").dropna(axis = 0, how = 'all')

In [111]:
# function to get country bigramme given currency trigramme
def cntry(curr):
    if currency['CURRENCY_CODE'].isin([curr]).any():
        return currency[currency['CURRENCY_CODE'] == curr].iat[0, 1]
    else:
        return 'US'

In [112]:
# function to shorten issuer name to 50 characters
def issuer(txt):
    if issuer_name['ISSUER_LONG_NAME'].isin([txt]).any():
        return issuer_name[issuer_name['ISSUER_LONG_NAME'] == txt].iat[0,1]
    else:
        return txt[:50]

# test the function
#text1 = 'TOM BURKE COMMUNITY TRUST INVESTMENT SPV (PTY) LTD RF'
#text2 = '123456789012345678901234567890123456789012345678901234567890123456789012345678901234567890'
#print(issuer(text1), len(text1), len(issuer(text1)))
#print(issuer(text2), len(text2), len(issuer(text2)))

In [113]:
# identify the empty entries in the "COUNTRY_CODE" column ("NA" for Namibia returns a NaN)
k = instruments.loc[(instruments['COUNTRY_CODE'].isnull())  &
                    (instruments['CURRENCY_CODE'] != 'NAD') &
                    (instruments['ISSUE_NAME']    != 'NAMIBIA')]

# identify the > 50 ISSUE_NAME securities in Instruments.csv
p = instruments[instruments['ISSUE_NAME'].str.len() > 50]

# identify the > 50 ISSUERCODE securities in Holdings.csv
q = holdings[holdings['ISSUERCODE'].str.len() > 50]

# identify the > 2**31 'PAR_OR_SHARES' Transactions.csv securities
r = transactions[transactions['PAR_OR_SHARES'] >= 2**31]

In [114]:
# (1) update missing instruments COUNTRY_CODE
if len(k) > 0:
    for row_number in range(len(k)):
        instruments.at[k.index[row_number], 'COUNTRY_CODE'] = cntry(instruments.at[k.index[row_number], 'CURRENCY_CODE'])

In [115]:
# (2) update long, i.e., > 50 characters, ISSUE_NAME in instruments dataframe
if len(p) > 0:
    for row_number in range(len(p)):
        instruments.at[p.index[row_number], 'ISSUE_NAME'] = issuer(instruments.at[p.index[row_number], 'ISSUE_NAME'])

In [116]:
# (3) update long, i.e., > 50 characters, ISSUERNAME in holdings dataframe
if len(q) > 0:
    for row_number in range(len(q)):
        holdings.at[q.index[row_number], 'ISSUERCODE'] = issuer(holdings.at[q.index[row_number], 'ISSUERCODE'])

In [123]:
# (4) update PAR_OR_SHARES values >= 2**31
if len(r) > 0:
    for row_number in range(len(r)):
        transactions.at[r.index[row_number], 'PAR_OR_SHARES'] = 2**31-1       

In [127]:
# save the datframes over the Instruments.csv and Holdings.csv files in the eagle folder
#https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.to_csv.html
instruments.to_csv( eagle_path + r'\Instruments.csv' , index  = False)
holdings.to_csv(    eagle_path + r'\Holdings.csv'    , index  = False)
transactions.to_csv(eagle_path + r'\Transactions.csv', index  = False)

In [122]:
print(f'Amending \eagle\.. files before StatPro run completed: {timediff(start_time, datetime.now())}')

Amending \eagle\.. files before StatPro run completed: 0d 0hr 1min 17.0sec
